# Dataset generation
Generates the training dataset by solving the Kirchhoff-Love FDM model on 21
substrate geometry classes (single-feature, multi-feature, curved-boundary, SWG).

In [ ]:
import numpy as np
import random
from itertools import product
from tqdm import tqdm

from fdm_solver import (
    get_w_strain,
    draw_circle,
    draw_rect,
    draw_ellipse,
    draw_parabola_region,
    apply_swg_mask,
)

In [ ]:
Ny, Nx = 256, 256
cx, cy = Nx // 2, Ny // 2

## Geometry classes

In [ ]:
SCENARIOS = {

    # ── Single-feature ───────────────────────────────────────────────────────
    'single_circle_center': lambda m: draw_circle(
        m, cx, cy, r=random.randint(20, 60)),

    'single_circle_edge': lambda m: draw_circle(
        m, random.choice([25, Nx - 25]), cy, r=random.randint(15, 35)),

    'single_rect_center': lambda m: draw_rect(
        m, cx, cy,
        w=random.randint(30, 100), h=random.randint(30, 100),
        angle_deg=random.uniform(0, 90)),

    'single_ellipse': lambda m: draw_ellipse(
        m, cx, cy,
        rx=random.randint(20, 70), ry=random.randint(10, 40),
        angle_deg=random.uniform(0, 180)),

    # ── Multi-feature ────────────────────────────────────────────────────────
    'two_circles_side': lambda m: [
        draw_circle(m, cx - random.randint(40, 70), cy, random.randint(15, 35)),
        draw_circle(m, cx + random.randint(40, 70), cy, random.randint(15, 35))],

    'two_circles_close': lambda m: [
        draw_circle(m, cx - random.randint(15, 30), cy, random.randint(15, 25)),
        draw_circle(m, cx + random.randint(15, 30), cy, random.randint(15, 25))],

    'circle_and_rect': lambda m: [
        draw_circle(m, cx - 40, cy, random.randint(15, 30)),
        draw_rect(m, cx + 40, cy,
                  random.randint(20, 50), random.randint(20, 50))],

    'grid_circles': lambda m: [
        draw_circle(m, x, y, random.randint(10, 20))
        for x, y in product(
            range(40, Nx - 40, random.randint(40, 70)),
            range(40, Ny - 40, random.randint(40, 70)))],

    'row_circles': lambda m: [
        draw_circle(m, x, cy, random.randint(10, 25))
        for x in range(40, Nx - 40, random.randint(35, 60))],

    'random_mix': lambda m: [
        (lambda shape, cx_, cy_:
            draw_circle(m, cx_, cy_, random.randint(10, 40))
            if shape == 'circle' else
            draw_ellipse(m, cx_, cy_, random.randint(10, 40),
                         random.randint(10, 30), random.uniform(0, 180))
         )(random.choice(['circle', 'ellipse']),
           random.randint(30, Nx - 30),
           random.randint(30, Ny - 30))
        for _ in range(random.randint(2, 6))],

    # ── Curved-boundary (parabolic) ──────────────────────────────────────────

    # concave-up (U-shaped) — filled below
    'parabola_concave_up': lambda m: draw_parabola_region(
        m, cx, cy,
        amplitude=random.uniform(0.2, 0.7),
        rotation_deg=0,
        fill='below'),

    # convex-up (∩-shaped) — filled above
    'parabola_convex_up': lambda m: draw_parabola_region(
        m, cx, cy,
        amplitude=random.uniform(0.2, 0.7),
        rotation_deg=0,
        fill='above'),

    # concave, arbitrary rotation
    'parabola_concave_rotated': lambda m: draw_parabola_region(
        m, cx, cy,
        amplitude=random.uniform(0.2, 0.7),
        rotation_deg=random.uniform(0, 360),
        fill='below'),

    # convex, arbitrary rotation
    'parabola_convex_rotated': lambda m: draw_parabola_region(
        m, cx, cy,
        amplitude=random.uniform(0.2, 0.7),
        rotation_deg=random.uniform(0, 360),
        fill='above'),

    # concave parabola + circle
    'parabola_and_circle': lambda m: [
        draw_parabola_region(
            m, cx, cy,
            amplitude=random.uniform(0.2, 0.7),
            rotation_deg=random.uniform(0, 360),
            fill=random.choice(['below', 'above'])),
        draw_circle(m,
                    random.randint(30, Nx - 30),
                    random.randint(30, Ny - 30),
                    random.randint(10, 25))],

    # parabola + two circles
    'parabola_and_2circles': lambda m: [
        draw_parabola_region(
            m, cx, cy,
            amplitude=random.uniform(0.2, 0.7),
            rotation_deg=random.uniform(0, 360),
            fill=random.choice(['below', 'above'])),
        draw_circle(m, cx - random.randint(20, 60), cy, random.randint(10, 20)),
        draw_circle(m, cx + random.randint(20, 60), cy, random.randint(10, 20))],

    # parabola + ellipse
    'parabola_and_ellipse': lambda m: [
        draw_parabola_region(
            m, cx, cy,
            amplitude=random.uniform(0.2, 0.7),
            rotation_deg=random.uniform(0, 360),
            fill=random.choice(['below', 'above'])),
        draw_ellipse(m,
                     random.randint(30, Nx - 30),
                     random.randint(30, Ny - 30),
                     rx=random.randint(10, 40),
                     ry=random.randint(10, 30),
                     angle_deg=random.uniform(0, 180))],

    # ── SWG ──────────────────────────────────────────────────────────────────
     'swg': lambda m: apply_swg_mask(m),
    # #18: SWG + parabola
     'swg_and_parabola': lambda m: [
        apply_swg_mask(m),                          # first draws the SWG pattern (m[:]=0, bars=1)
        draw_parabola_region(                        # then carves out the parabola
            m, cx, cy,
            amplitude=random.uniform(0.2, 0.7),
            rotation_deg=random.uniform(0, 360),
            fill=random.choice(['below', 'above']))],

    # #19: SWG + circle
     'swg_and_circle': lambda m: [
        apply_swg_mask(m),
        draw_circle(m,
                    random.randint(30, Nx - 30),
                    random.randint(30, Ny - 30),
                    random.randint(15, 50))],

    # #20: SWG + ellipse
    'swg_and_ellipse': lambda m: [
        apply_swg_mask(m),
        draw_ellipse(m,
                     random.randint(30, Nx - 30),
                     random.randint(30, Ny - 30),
                     rx=random.randint(15, 55),
                     ry=random.randint(10, 35),
                     angle_deg=random.uniform(0, 180))],
}

## Sample generation wrapper

In [ ]:
def generate_sample(scenario_name, width=6e-6):
    mask = np.zeros((Ny, Nx), dtype=int)
    SCENARIOS[scenario_name](mask)
    strain, X_coord, Y_coord = get_w_strain(mask, width)
    return mask, strain * 100, X_coord, Y_coord

## Build and save the dataset

In [ ]:
N_per_scenario = 300
scenario_names = list(SCENARIOS.keys())
X_data = []
Z_data = []
W_data = []
M_data = []
for scenario in scenario_names:
    print(f"Generating: {scenario}")
    for _ in tqdm(range(N_per_scenario)):
        mask, strain, X_coord, Y_coord = generate_sample(scenario, 12e-6)
        X_data.append(mask.astype(np.float32))
        Z_data.append(strain.astype(np.float32))
        W_data.append(X_coord.astype(np.float32))
        M_data.append(Y_coord.astype(np.float32))

X_data = np.array(X_data)[:, np.newaxis]
Z_data = np.array(Z_data)[:, np.newaxis]
W_data = np.array(W_data)[:, np.newaxis]
M_data = np.array(M_data)[:, np.newaxis]


np.savez_compressed('dataset_12mkm_2.npz', support_map = X_data, strain = Z_data, X_coord = W_data, Y_coord = M_data)
print(f"Dataset: {X_data.shape}, saved.")